# 💬 Colab 3 — Prompt Engineering en RAG
### Materia: Procesamiento de Lenguaje Natural — Nivel Terciario

---

## La pregunta central de este colab

> ¿Qué pasa si el retrieval es perfecto pero el prompt es malo?

En los colabs anteriores nos enfocamos en **qué chunks recuperar**. Ahora el retrieval está fijo y la variable es **cómo le pedimos al LLM que use ese contexto**.

Esto sorprende a muchos: es posible tener el contexto correcto y aun así obtener una respuesta incorrecta, incompleta o inutilizable, solo por cómo está redactado el prompt.

## Las variables que vamos a explorar

| Variable | Descripción |
|---|---|
| **Sin instrucciones** | Solo contexto + pregunta, sin sistema |
| **Instrucciones vagas** | "Respondé usando el contexto" |
| **Instrucciones precisas** | Rol, restricciones, comportamiento ante ausencia de info |
| **Formato de salida** | JSON, lista, respuesta corta, respuesta extensa |
| **Manejo de ausencia** | Qué hace el modelo cuando la info no está en el contexto |
| **Chain of Thought** | Pedirle que razone paso a paso antes de responder |

> 🎯 **Objetivo:** entender que el prompt es una decisión de diseño tan importante como el chunking o el modelo elegido.

In [1]:
!pip install chromadb openai --quiet
print('✅ Dependencias instaladas')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the so

In [2]:
import os
from openai import OpenAI

try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
except Exception:
    api_key = os.getenv('OPENAI_API_KEY', 'TU_API_KEY_ACA')

client = OpenAI(api_key=api_key)
print('✅ Cliente OpenAI inicializado')

✅ Cliente OpenAI inicializado


---
## 📄 Setup: corpus y retrieval fijos

En este colab el retrieval **no cambia**. Siempre recuperamos los mismos chunks.
La única variable es el prompt que armamos con ese contexto.

In [3]:
import chromadb
from chromadb.utils import embedding_functions

CHUNKS = [
    "Los clientes tienen derecho a solicitar la devolución de productos dentro de los 30 días corridos desde la fecha de entrega. Para productos con defectos de fábrica, el plazo se extiende a 90 días. Las devoluciones fuera de estos plazos no serán aceptadas salvo orden judicial.",
    "El producto debe presentarse en su embalaje original, sin señales de uso, con todos los accesorios incluidos y con el ticket o factura de compra. Los productos de higiene personal, auriculares in-ear y colchones no admiten devolución por razones sanitarias.",
    "El cliente debe iniciar el proceso de devolución a través del portal web en la sección Mi Cuenta. Una vez aprobada la solicitud recibirá un código RMA. Las devoluciones sin código RMA serán rechazadas. El reembolso se acredita dentro de los 10 días hábiles.",
    "Todos los productos tienen garantía legal mínima de 6 meses por defectos de fabricación, conforme a la Ley 24.240 de Defensa del Consumidor.",
    "Los televisores de 40 pulgadas o más tienen garantía de 12 meses. Las notebooks, tablets y computadoras tienen garantía de 12 meses. Los smartphones de gama alta tienen garantía de 12 meses. Los accesorios y cables tienen garantía de 3 meses.",
    "La garantía no cubre daños causados por mal uso, golpes, líquidos, modificaciones no autorizadas o uso fuera de las condiciones del fabricante. Los daños por sobretensión solo están cubiertos con estabilizador certificado.",
    "Para hacer efectiva la garantía el cliente debe presentar el producto en cualquier sucursal. El tiempo de reparación estimado es de 15 días hábiles. Si no puede completarse, TechStore ofrecerá reemplazo o reembolso.",
    "El envío es gratuito para compras iguales o superiores a $50.000. Para compras menores el costo se calcula según peso y distancia. Los productos de más de 30 kg tienen cargo adicional.",
    "Para CABA y GBA el plazo es de 24 a 48 horas hábiles. Para el interior es de 3 a 7 días hábiles. Patagonia y NOA tienen un plazo de 7 a 12 días hábiles.",
    "Se aceptan tarjetas Visa, Mastercard, American Express y Cabal. También billeteras digitales: Mercado Pago, Modo, Ualá y Naranja X. Las tarjetas prepagas solo se aceptan hasta $30.000.",
    "Con Visa y Mastercard se ofrecen 3, 6 y 12 cuotas sin interés en productos seleccionados. Los productos en liquidación no participan. Las cuotas no son acumulables con otros descuentos.",
    "Los pagos por transferencia bancaria reciben un descuento adicional del 10%. La transferencia debe acreditarse dentro de las 48 horas o la orden se cancela.",
    "El centro de atención opera de lunes a viernes de 9 a 18 hs y sábados de 9 a 13 hs. Canales: WhatsApp +54 11 4000-0000, email soporte@techstore.com.ar, chat en vivo y atención presencial.",
    "Los reclamos formales se resuelven en un máximo de 72 horas hábiles. En casos complejos el plazo puede extenderse hasta 10 días hábiles notificando al cliente.",
    "Si el cliente no está satisfecho puede escalar al área de Supervisión o presentar el reclamo ante la Dirección de Defensa del Consumidor de su provincia.",
]

openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=api_key,
    model_name="text-embedding-3-small"
)
chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("colab3")
except:
    pass
coleccion = chroma_client.create_collection("colab3", embedding_function=openai_ef)
coleccion.add(documents=CHUNKS, ids=[f"c{i}" for i in range(len(CHUNKS))])

def recuperar_contexto(pregunta, top_k=3):
    res = coleccion.query(query_texts=[pregunta], n_results=top_k)
    return "\n\n".join(res['documents'][0])

def llamar_llm(system_prompt, user_prompt, temperature=0.3, max_tokens=300):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content, response.usage.total_tokens

print(f"✅ Vector store listo con {coleccion.count()} chunks")
print("   El retrieval está fijo. Solo vamos a cambiar el prompt.")

✅ Vector store listo con 15 chunks
   El retrieval está fijo. Solo vamos a cambiar el prompt.


---
# 🔬 Experimento 1 — Sin instrucciones de sistema

El caso más básico: pasamos el contexto y la pregunta sin ninguna instrucción adicional.
¿Qué hace el modelo cuando no tiene guía?

In [4]:
PREGUNTA = "¿Cuánto tiempo tengo para devolver un producto y qué condiciones tiene?"

contexto = recuperar_contexto(PREGUNTA)

# Sin system prompt, solo contexto + pregunta en el mensaje de usuario
user_prompt_sin_instrucciones = f"""{contexto}

{PREGUNTA}"""

respuesta, tokens = llamar_llm(
    system_prompt=None,
    user_prompt=user_prompt_sin_instrucciones
)

print("⚙️  Configuración: SIN instrucciones de sistema")
print(f"Tokens usados: {tokens}")
print("=" * 60)
print(respuesta)

⚙️  Configuración: SIN instrucciones de sistema
Tokens usados: 318
Tienes 30 días corridos desde la fecha de entrega para devolver un producto. Si el producto tiene defectos de fábrica, el plazo se extiende a 90 días. Las devoluciones fuera de estos plazos no serán aceptadas, salvo orden judicial.

Para iniciar el proceso de devolución, debes hacerlo a través del portal web en la sección "Mi Cuenta". Una vez que tu solicitud sea aprobada, recibirás un código RMA, el cual es necesario para realizar la devolución, ya que las devoluciones sin este código serán rechazadas. El reembolso se acreditará dentro de los 10 días hábiles después de que se procese la devolución.


### 🔍 ¿Qué observás?

Sin instrucciones, el modelo responde pero con comportamiento impredecible:
- Puede responder en cualquier tono (formal, informal)
- Puede inventar información más allá del contexto
- Puede ignorar parte del contexto
- El formato es arbitrario

Esto ya es mejor que sin RAG, pero todavía no es confiable para producción.

---
# 🔬 Experimento 2 — Instrucciones vagas vs precisas

Comparamos tres niveles de especificidad en el system prompt.

In [5]:
PREGUNTA = "¿Cuánto tiempo tengo para devolver un producto y qué condiciones tiene?"
contexto = recuperar_contexto(PREGUNTA)

# ── Nivel 1: Instrucción vaga ──
system_vago = "Respondé usando el contexto provisto."

# ── Nivel 2: Instrucción intermedia ──
system_intermedio = """Sos un asistente de atención al cliente.
Respondé basándote en el contexto provisto.
Si no tenés la información, decilo."""

# ── Nivel 3: Instrucción precisa ──
system_preciso = """Sos Tomi, el asistente virtual de TechStore Argentina.
Tu rol es responder consultas de clientes de forma clara, amable y profesional.

REGLAS:
- Respondé ÚNICAMENTE con información del contexto provisto
- Si la información no está en el contexto, decí exactamente: "No tengo esa información disponible. Te recomiendo contactarnos por WhatsApp al +54 11 4000-0000."
- No inventes datos, plazos, precios ni condiciones
- Usá un tono amigable pero profesional
- Respondé siempre en español argentino (tuteo)
- Si la respuesta tiene múltiples condiciones, usá una lista para mayor claridad"""

user_prompt = f"""CONTEXTO:
{contexto}

CONSULTA DEL CLIENTE:
{PREGUNTA}"""

niveles = [
    ("Vago",       system_vago),
    ("Intermedio", system_intermedio),
    ("Preciso",    system_preciso),
]

for nombre, system in niveles:
    respuesta, tokens = llamar_llm(system, user_prompt)
    print(f"\n🎚️  System prompt {nombre} (tokens: {tokens}):")
    print("-" * 60)
    print(respuesta)
    print()


🎚️  System prompt Vago (tokens: 283):
------------------------------------------------------------
Tienes 30 días corridos desde la fecha de entrega para devolver un producto. Si el producto tiene defectos de fábrica, el plazo se extiende a 90 días. Es importante que inicies el proceso de devolución a través del portal web en la sección Mi Cuenta y que obtengas un código RMA, ya que las devoluciones sin este código serán rechazadas.


🎚️  System prompt Intermedio (tokens: 325):
------------------------------------------------------------
Tienes 30 días corridos desde la fecha de entrega para devolver un producto. Si el producto tiene defectos de fábrica, el plazo se extiende a 90 días. Es importante que inicies el proceso de devolución a través del portal web en la sección Mi Cuenta. Una vez que tu solicitud sea aprobada, recibirás un código RMA, el cual es necesario para realizar la devolución. Ten en cuenta que las devoluciones sin código RMA serán rechazadas.


🎚️  System prompt Pr

### 🔍 ¿Qué diferencias notás?

Observá especialmente:
- ¿El modelo se mantiene dentro del contexto o agrega información propia?
- ¿El tono es consistente con la identidad de la empresa?
- ¿Cómo maneja las condiciones múltiples (las lista, las mezcla, las omite)?
- ¿El prompt preciso da más tokens? ¿Vale la pena el costo extra?

---
# 🔬 Experimento 3 — Control del formato de salida

El mismo contexto y la misma pregunta, pero le pedimos distintos formatos.
Esto es crítico cuando la respuesta del LLM se va a procesar programáticamente.

In [6]:
PREGUNTA = "¿Qué garantía tienen los productos y cuáles son las exclusiones?"
contexto = recuperar_contexto(PREGUNTA)

system_base = """Sos un asistente de atención al cliente de TechStore Argentina.
Respondé ÚNICAMENTE con información del contexto provisto.
No inventes datos."""

formatos = {
    "Respuesta libre": """{contexto}

PREGUNTA: {pregunta}""",

    "Respuesta corta (máx 2 oraciones)": """{contexto}

PREGUNTA: {pregunta}

Respondé en máximo 2 oraciones, solo lo más importante.""",

    "Lista de puntos": """{contexto}

PREGUNTA: {pregunta}

Respondé en formato de lista de puntos (bullet points), uno por condición relevante.""",

    "JSON estructurado": """{contexto}

PREGUNTA: {pregunta}

Respondé ÚNICAMENTE con un JSON válido con esta estructura, sin texto adicional:
{{
  "garantia_minima": "...",
  "garantias_extendidas": {{"producto": "plazo"}},
  "exclusiones": ["...", "..."],
  "proceso": "..."
}}""",

    "Respuesta para WhatsApp": """{contexto}

PREGUNTA: {pregunta}

Respondé como si fuera un mensaje de WhatsApp: informal, con emojis relevantes, máximo 3 líneas.""",
}

for nombre, template in formatos.items():
    user_prompt = template.format(contexto=contexto, pregunta=PREGUNTA)
    respuesta, tokens = llamar_llm(system_base, user_prompt, max_tokens=400)
    print(f"\n📋 Formato: {nombre} (tokens: {tokens})")
    print("-" * 60)
    print(respuesta)
    print()


📋 Formato: Respuesta libre (tokens: 324)
------------------------------------------------------------
Todos los productos tienen una garantía legal mínima de 6 meses por defectos de fabricación, conforme a la Ley 24.240 de Defensa del Consumidor. Sin embargo, hay excepciones:

- Los televisores de 40 pulgadas o más, las notebooks, tablets y computadoras, así como los smartphones de gama alta tienen una garantía de 12 meses.
- Los accesorios y cables tienen una garantía de 3 meses.

Las exclusiones de la garantía son los daños causados por mal uso, golpes, líquidos, modificaciones no autorizadas o uso fuera de las condiciones del fabricante. Además, los daños por sobretensión solo están cubiertos si se utiliza un estabilizador certificado.


📋 Formato: Respuesta corta (máx 2 oraciones) (tokens: 288)
------------------------------------------------------------
Todos los productos tienen una garantía legal mínima de 6 meses por defectos de fabricación, con excepciones como los televisore

### 🔍 Por qué el formato importa en producción

- **JSON:** si el frontend necesita procesar la respuesta, el formato libre rompe todo
- **Corto:** para chatbots de WhatsApp, una respuesta larga ahuyenta al cliente
- **Lista:** para condiciones múltiples, facilita la lectura y reduce malentendidos
- **Informal:** el tono debe coincidir con el canal de comunicación

> 💡 En sistemas RAG reales se suele definir el formato de salida según el canal de destino: app web, WhatsApp, email, API interna.

---
# 🔬 Experimento 4 — Manejo de información ausente

¿Qué pasa cuando el cliente pregunta algo que no está en el corpus?
Este es uno de los comportamientos más críticos de un sistema RAG en producción.

In [7]:
# Preguntas cuya respuesta NO está en el corpus
preguntas_sin_respuesta = [
    "¿Tienen sucursal en Mendoza?",
    "¿Puedo retirar mi compra en el día en el local de Palermo?",
    "¿Hacen descuentos para jubilados?",
]

system_sin_control = "Sos un asistente de TechStore Argentina. Respondé las consultas de los clientes."

system_con_control = """Sos un asistente de TechStore Argentina.
Respondé ÚNICAMENTE con información del contexto provisto.
Si la información no está en el contexto, respondé EXACTAMENTE:
"Esa información no está disponible en este momento. Para consultas específicas contactanos por WhatsApp al +54 11 4000-0000 o por email a soporte@techstore.com.ar."
No agregues suposiciones ni información propia."""

for pregunta in preguntas_sin_respuesta:
    contexto = recuperar_contexto(pregunta)
    user_prompt = f"CONTEXTO:\n{contexto}\n\nCONSULTA: {pregunta}"

    resp_sin, _ = llamar_llm(system_sin_control, user_prompt)
    resp_con, _ = llamar_llm(system_con_control, user_prompt)

    print(f"\n❓ '{pregunta}'")
    print("=" * 60)
    print(f"🔴 Sin control de alucinación:")
    print(f"   {resp_sin[:200]}")
    print(f"\n🟢 Con control de alucinación:")
    print(f"   {resp_con[:200]}")
    print()


❓ '¿Tienen sucursal en Mendoza?'
🔴 Sin control de alucinación:
   Actualmente, no contamos con sucursales físicas en Mendoza. Sin embargo, puedes realizar tus compras a través de nuestra tienda online y recibir tus productos en la dirección que elijas. Si necesitas 

🟢 Con control de alucinación:
   Esa información no está disponible en este momento. Para consultas específicas contactanos por WhatsApp al +54 11 4000-0000 o por email a soporte@techstore.com.ar.


❓ '¿Puedo retirar mi compra en el día en el local de Palermo?'
🔴 Sin control de alucinación:
   Sí, puedes retirar tu compra en el local de Palermo el mismo día, siempre y cuando el producto esté disponible para retiro. Te recomendamos que realices la compra en línea y elijas la opción de "retir

🟢 Con control de alucinación:
   Esa información no está disponible en este momento. Para consultas específicas contactanos por WhatsApp al +54 11 4000-0000 o por email a soporte@techstore.com.ar.


❓ '¿Hacen descuentos para jubilados

### 🔍 Este experimento muestra algo crítico

Sin control explícito en el prompt, el modelo **inventa** sucursales, horarios y políticas que no existen. Lo hace con total confianza.

Con la instrucción precisa de "si no está en el contexto, decí X", el comportamiento es predecible y controlado.

> ⚠️ En un sistema de atención al cliente real, una alucinación sobre horarios o sucursales puede generar un cliente enojado que viajó en vano.

---
# 🔬 Experimento 5 — Chain of Thought (razonamiento paso a paso)

Para preguntas complejas que requieren combinar información de múltiples chunks,
pedirle al modelo que razone antes de responder puede mejorar la calidad.

In [8]:
# Pregunta que requiere combinar múltiples piezas de información
PREGUNTA_COMPLEJA = """Compré una notebook hace 8 meses y dejó de funcionar.
¿Tengo garantía? ¿Qué tengo que hacer y cuánto tarda el proceso?"""

contexto = recuperar_contexto(PREGUNTA_COMPLEJA, top_k=4)

system_base = """Sos un asistente de TechStore Argentina.
Respondé ÚNICAMENTE con información del contexto provisto."""

# Sin chain of thought
user_directo = f"""CONTEXTO:
{contexto}

CONSULTA: {PREGUNTA_COMPLEJA}"""

# Con chain of thought
user_cot = f"""CONTEXTO:
{contexto}

CONSULTA: {PREGUNTA_COMPLEJA}

Antes de responder, analizá paso a paso:
1. ¿Qué tipo de producto es y qué garantía le corresponde?
2. ¿El tiempo transcurrido está dentro del plazo de garantía?
3. ¿Qué pasos debe seguir el cliente?
4. ¿Hay alguna exclusión que podría aplicar?

Luego de tu análisis, dá una respuesta clara y concisa al cliente."""

resp_directo, tok_directo = llamar_llm(system_base, user_directo, max_tokens=400)
resp_cot,     tok_cot     = llamar_llm(system_base, user_cot,     max_tokens=500)

print("⚡ Respuesta directa (sin CoT):")
print(f"   Tokens: {tok_directo}")
print("-" * 60)
print(resp_directo)

print("\n\n🧠 Respuesta con Chain of Thought:")
print(f"   Tokens: {tok_cot}")
print("-" * 60)
print(resp_cot)

⚡ Respuesta directa (sin CoT):
   Tokens: 404
------------------------------------------------------------
Sí, tienes garantía para tu notebook, ya que cuenta con 12 meses de garantía. Para hacer efectiva la garantía, debes presentar el producto en cualquier sucursal de TechStore. 

El proceso que debes seguir es el siguiente:
1. Iniciar el proceso de devolución a través del portal web en la sección Mi Cuenta.
2. Una vez aprobada la solicitud, recibirás un código RMA.

Ten en cuenta que las devoluciones sin código RMA serán rechazadas. El tiempo estimado de reparación es de 15 días hábiles. Si no se puede completar la reparación, TechStore te ofrecerá un reemplazo o reembolso. El reembolso se acredita dentro de los 10 días hábiles.


🧠 Respuesta con Chain of Thought:
   Tokens: 493
------------------------------------------------------------
1. Compraste una notebook, que tiene una garantía de 12 meses.
2. El tiempo transcurrido es de 8 meses, lo que está dentro del plazo de garantía.


### 🔍 Trade-off del Chain of Thought

**Ventajas:**
- Respuestas más precisas en preguntas que requieren razonamiento multi-paso
- Menos probabilidad de ignorar condiciones relevantes
- El "razonamiento" visible ayuda a debuggear el sistema

**Desventajas:**
- Más tokens = más costo y más latencia
- Para preguntas simples no agrega valor

> 💡 En producción se suele usar CoT solo para consultas clasificadas como "complejas", no para todas.

---
# 🔬 Experimento 6 — El prompt completo de producción

Juntamos todo lo que aprendimos en un único prompt de producción robusto.

In [9]:
SYSTEM_PRODUCCION = """Sos Tomi, el asistente virtual de TechStore Argentina.
Tu objetivo es resolver las consultas de los clientes de forma clara, precisa y amable.

INSTRUCCIONES:
- Respondé ÚNICAMENTE con información del contexto provisto
- Si la información no está en el contexto, respondé: "Esa consulta requiere atención personalizada. Contactanos por WhatsApp al +54 11 4000-0000 (lunes a viernes 9-18 hs)."
- Usá español argentino con tuteo
- Sé conciso: máximo 4 oraciones o una lista de hasta 5 puntos
- Si hay condiciones o pasos a seguir, usá una lista numerada
- Nunca menciones que tenés un "contexto" o que sos una IA

TONO: Amigable, profesional, directo. Como un vendedor experimentado que conoce bien los productos."""

# Probamos con distintas preguntas
preguntas_finales = [
    "Compré un televisor de 55 pulgadas hace 10 meses y no enciende. ¿Qué hago?",
    "¿Puedo pagar en 12 cuotas sin interés con tarjeta Naranja?",
    "¿Tienen envío express para el mismo día?",  # info no disponible en corpus
    "Me llegó el pedido y falta un accesorio. ¿Cómo reclamo?",
]

for pregunta in preguntas_finales:
    contexto = recuperar_contexto(pregunta, top_k=3)
    user_prompt = f"CONTEXTO:\n{contexto}\n\nCLIENTE: {pregunta}"
    respuesta, tokens = llamar_llm(SYSTEM_PRODUCCION, user_prompt, temperature=0.2)

    print(f"\n❓ {pregunta}")
    print(f"   Tokens: {tokens}")
    print(f"   Tomi: {respuesta}")
    print("-" * 60)


❓ Compré un televisor de 55 pulgadas hace 10 meses y no enciende. ¿Qué hago?
   Tokens: 444
   Tomi: Para hacer efectiva la garantía de tu televisor de 55 pulgadas, seguí estos pasos:

1. Presentá el televisor en cualquier sucursal de TechStore.
2. Lleva el comprobante de compra.
3. El tiempo de reparación estimado es de 15 días hábiles. Si no se puede reparar, te ofreceremos un reemplazo o reembolso.

Recuerda que tu televisor tiene garantía de 12 meses.
------------------------------------------------------------

❓ ¿Puedo pagar en 12 cuotas sin interés con tarjeta Naranja?
   Tokens: 378
   Tomi: No, no se aceptan tarjetas Naranja para el pago en cuotas sin interés. Solo se aceptan Visa y Mastercard para esa opción. Podés utilizar Naranja para otros métodos de pago, pero no para cuotas.
------------------------------------------------------------

❓ ¿Tienen envío express para el mismo día?
   Tokens: 366
   Tomi: Esa consulta requiere atención personalizada. Contactanos por WhatsAp

---
# 🧪 Zona de experimentación libre

In [10]:
# ✏️ Armá tu propio system prompt y probalo

MI_SYSTEM_PROMPT = """
Escribí acá tu propio system prompt.
Probá cambiar el tono, el nombre del asistente, las restricciones, el formato...
"""

MI_PREGUNTA = "¿Cuánto tarda en llegar un pedido a Córdoba?"

contexto = recuperar_contexto(MI_PREGUNTA)
user_prompt = f"CONTEXTO:\n{contexto}\n\nCONSULTA: {MI_PREGUNTA}"

respuesta, tokens = llamar_llm(MI_SYSTEM_PROMPT, user_prompt)
print(f"Tokens: {tokens}")
print("=" * 60)
print(respuesta)

Tokens: 216
Un pedido a Córdoba tarda entre 3 a 7 días hábiles en llegar. Si tienes más preguntas o necesitas información adicional, ¡no dudes en preguntar!


In [11]:
# 🔬 Comparador rápido: probá dos prompts en paralelo

PROMPT_A = """Sos un asistente formal y técnico. Respondé con precisión usando solo el contexto."""
PROMPT_B = """Sos un asistente amigable. Respondé de forma simple y cercana usando solo el contexto."""

PREGUNTA_TEST = "¿Qué pasa si me arrepiento de una compra?"

contexto = recuperar_contexto(PREGUNTA_TEST)
user_prompt = f"CONTEXTO:\n{contexto}\n\nPREGUNTA: {PREGUNTA_TEST}"

resp_a, tok_a = llamar_llm(PROMPT_A, user_prompt)
resp_b, tok_b = llamar_llm(PROMPT_B, user_prompt)

print(f"❓ Pregunta: {PREGUNTA_TEST}\n")
print(f"🅰️  Prompt A (tokens: {tok_a}):")
print(resp_a)
print(f"\n🅱️  Prompt B (tokens: {tok_b}):")
print(resp_b)

❓ Pregunta: ¿Qué pasa si me arrepiento de una compra?

🅰️  Prompt A (tokens: 317):
Si te arrepientes de una compra, puedes solicitar la devolución del producto dentro de los 30 días corridos desde la fecha de entrega. Debes iniciar el proceso a través del portal web en la sección Mi Cuenta. Una vez aprobada la solicitud, recibirás un código RMA, que es necesario para realizar la devolución. Ten en cuenta que las devoluciones sin código RMA serán rechazadas. Si el producto tiene defectos de fábrica, el plazo para la devolución se extiende a 90 días. Las devoluciones fuera de estos plazos no serán aceptadas, salvo orden judicial.

🅱️  Prompt B (tokens: 275):
Si te arrepientes de una compra, puedes solicitar la devolución del producto dentro de los 30 días corridos desde la fecha de entrega. Solo debes iniciar el proceso a través del portal web en la sección Mi Cuenta. Recuerda que necesitarás un código RMA para que la devolución sea aceptada. Si tienes algún problema, puedes escalarlo al

---
# ✅ Resumen del Colab 3

| Técnica | Cuándo usarla | Costo |
|---|---|---|
| **Sin instrucciones** | Nunca en producción | Bajo |
| **Instrucción precisa con rol** | Siempre como base | Bajo |
| **Control de alucinación explícito** | Siempre | Sin costo extra |
| **Formato estructurado (JSON, lista)** | Cuando el output se procesa | Sin costo extra |
| **Chain of Thought** | Preguntas complejas multi-paso | Alto (más tokens) |

## 🔑 Las tres reglas del prompt en RAG

1. **Definí el rol con precisión:** nombre, empresa, tono, restricciones
2. **Controlá el comportamiento ante ausencia de información:** es el error más costoso en producción
3. **Especificá el formato de salida:** no lo dejes librado al modelo

## 🚀 Siguiente: Colab 4 — Modelos y Temperature
Mismo retrieval, mismo prompt. Ahora cambiamos el modelo y la temperature y vemos el impacto en calidad, costo y latencia.